In [ ]:
# pip install pybaseball pandas numpy plotly

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
from pybaseball import statcast
from scipy.spatial.distance import pdist

In [4]:
from pybaseball import statcast
df = statcast('2025-03-20', '2025-10-01')
# Rangers when playing at home
tex_home = df[df['home_team'] == 'TEX']

# Rangers when playing away
tex_away = df[df['away_team'] == 'TEX']

# Combine both
tex = pd.concat([tex_home, tex_away])

This is a large query, it may take a moment to complete


/Users/justinrodnick/anaconda3/lib/python3.11/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|█████████████████████████████████████████| 196/196 [00:52<00:00,  3.70it/s]


In [6]:
# Convert pitch movement from feet to inches
# Horizontal break (hb) is pfx_x * 12
# Induced vertical break (ivb) is -pfx_z * 12 (negative to match convention)
df['hb'] = df['pfx_x'] * 12
df['ivb'] = -df['pfx_z'] * 12

# Create a 'whiff' flag (True/False)
# A whiff occurs when the description is a swinging strike or a swinging strike that was blocked
df["whiff"] = df["description"].isin(["swinging_strike", "swinging_strike_blocked"])

# Create a 'called' flag (True/False)
# Marks pitches that were called strikes
df["called"] = df["description"].isin(["called_strike"])

# Calculate CSW (Called Strikes + Whiffs)
# CSW is True if the pitch resulted in either a whiff or a called strike
df["csw"] = df["whiff"] | df["called"]

In [7]:
# This filters df so that it only includes these columns
tex = df.filter([
    'pitcher',
    'player_name',
    'pitch_name',
    'release_speed',
    'pfx_x',
    'pfx_z',
    'estimated_woba_using_speedangle',
    'description',
    'hb',
    'ivb',
    'whiff',
    'called',
    'csw'
])

In [8]:
print(tex.columns)

Index(['pitcher', 'player_name', 'pitch_name', 'release_speed', 'pfx_x',
       'pfx_z', 'estimated_woba_using_speedangle', 'description', 'hb', 'ivb',
       'whiff', 'called', 'csw'],
      dtype='object')


In [13]:
# Define a function to compute Shannon entropy
def shannon_entropy(s):
    # Calculate the probability distribution of the values in the Series
    p = s.value_counts(normalize=True)
    # Apply the Shannon entropy formula: -Σ(p * log(p))
    return -(p * np.log(p)).sum()

# Group the DataFrame 'tex' by 'pitcher' and 'player_name'
arsenal = tex.groupby(["pitcher", "player_name"]).apply(
    # For each group, apply a custom function
    lambda g: pd.Series({
        # Count the number of pitches in the group
        "pitches": len(g),
        # Calculate the entropy of the 'pitch_name' column for that group
        "entropy": shannon_entropy(g["pitch_name"])
    })
# Reset the index so the grouped columns become normal columns again
).reset_index()

In [15]:
# Define a function to calculate velocity gaps between pitch types for each pitcher
def vel_gap(group):
    # Calculate the average fastball velocity (including Fastball, Sinker, Cutter)
    fb = group.loc[group["pitch_name"].str.contains("Fastball|Sinker|Cutter", na=False), "release_speed"].mean()
    
    # Calculate the average changeup velocity
    ch = group.loc[group["pitch_name"].str.contains("Change", na=False), "release_speed"].mean()
    
    # Calculate the average slider velocity
    sl = group.loc[group["pitch_name"].str.contains("Slider", na=False), "release_speed"].mean()
    
    # Calculate the average curveball velocity
    cb = group.loc[group["pitch_name"].str.contains("Curve", na=False), "release_speed"].mean()
    
    # Return a pandas Series with the calculated velocity values and differences
    return pd.Series({
        "fb_vel": fb,  # Average fastball velocity
        
        # Fastball - Changeup velocity gap (only if both exist, else NaN)
        "gap_FB_CH": fb - ch if pd.notna(ch) else np.nan,
        
        # Fastball - Slider velocity gap
        "gap_FB_SL": fb - sl if pd.notna(sl) else np.nan,
        
        # Fastball - Curveball velocity gap
        "gap_FB_CB": fb - cb if pd.notna(cb) else np.nan,
    })

# Apply the vel_gap function to each group of pitcher and player_name
# This computes the mean velocity and gaps per pitcher
# .reset_index() flattens the resulting grouped DataFrame for easy viewing
vel = tex.groupby(["pitcher", "player_name"]).apply(vel_gap).reset_index()

In [16]:
# Convert to numeric and drop missing values
df["ivb"] = pd.to_numeric(df["ivb"], errors="coerce")
df["hb"] = pd.to_numeric(df["hb"], errors="coerce")
df = df.dropna(subset=["ivb", "hb"])

# Group by pitcher, player, and pitch type
mov = df.groupby(["pitcher", "player_name", "pitch_name"]).agg(
    ivb=("ivb", "mean"),
    hb=("hb", "mean")
).reset_index()

# Function to get mean pairwise distance between pitches
def mean_pairwise_distance(group):
    if len(group) < 2:
        return np.nan
    data = group[["ivb", "hb"]].astype(float).to_numpy()
    return pdist(data, metric="euclidean").mean()

# Apply to each pitcher and player group
move_sep = (
    mov.groupby(["pitcher", "player_name"])
       .apply(mean_pairwise_distance)
       .reset_index(name="move_sep")
)

print(move_sep.head())

   pitcher        player_name   move_sep
0   434378  Verlander, Justin  20.051661
1   445276     Jansen, Kenley  19.034972
2   445926      Chavez, Jesse  21.356248
3   448179         Hill, Rich  21.475061
4   450203    Morton, Charlie  20.581779


In [18]:
# Group the DataFrame `tex` by pitcher and player name
eff = tex.groupby(["pitcher", "player_name"]).agg(
    # Calculate the mean of 'estimated_woba_using_speedangle' for each group, label it 'xwOBA'
    xwOBA=("estimated_woba_using_speedangle", "mean"),
    
    # Calculate the mean of 'whiff' for each group, label it 'whiff_rate'
    whiff_rate=("whiff", "mean"),
    
    # Calculate the mean of 'csw' (called strikes + whiffs) for each group, label it 'csw_rate'
    csw_rate=("csw", "mean")
# Reset the index so 'pitcher' and 'player_name' become columns again
).reset_index()

In [21]:
# Group the 'vel' DataFrame by pitcher and player name, 
# take the mean of all numeric columns within each group, 
# and then reset the index so 'pitcher' and 'player_name' become columns again
vel = vel.groupby(["pitcher", "player_name"]).mean().reset_index()

# Do the same grouping and averaging for the 'move_sep' DataFrame
move_sep = move_sep.groupby(["pitcher", "player_name"]).mean().reset_index()

# Do the same grouping and averaging for the 'eff' DataFrame
eff = eff.groupby(["pitcher", "player_name"]).mean().reset_index()

In [23]:
# Merge the 'arsenal' DataFrame with 'vel' on the columns 'pitcher' and 'player_name'
summary = (
    arsenal
    .merge(vel, on=['pitcher', 'player_name'])  # Merge velocity data
    .merge(move_sep, on=['pitcher', 'player_name'])  # Merge movement/separation data
    .merge(eff, on=['pitcher', 'player_name'])  # Merge effectiveness metrics
)

# Display the final merged DataFrame
summary

,pitcher,player_name,pitches,entropy,fb_vel,gap_FB_CH,gap_FB_SL,gap_FB_CB,move_sep,xwOBA,whiff_rate,csw_rate
0,434378,"Verlander, Justin",2693.0,1.402453,93.915856,9.237027,6.830374,15.412247,20.051661,0.325886,0.108429,0.277014
1,445276,"Jansen, Kenley",916.0,0.658114,92.750964,NaN,9.168424,NaN,19.034972,0.310111,0.118996,0.258734
2,445926,"Chavez, Jesse",230.0,1.330049,88.591367,4.805653,7.877081,14.382276,21.356248,0.393942,0.056522,0.239130
3,448179,"Hill, Rich",171.0,1.399076,87.095050,6.595050,NaN,14.681490,21.475061,0.398162,0.076023,0.269006
4,450203,"Morton, Charlie",2594.0,1.454942,93.068385,5.552180,NaN,11.628324,20.581779,0.339672,0.116423,0.279491
...,...,...,...,...,...,...,...,...,...,...,...,...
1013,823996,"Craig, Luke",31.0,0.602440,92.322727,NaN,11.444949,NaN,24.046274,<NA>,0.032258,0.096774
1014,827744,"Bryant, Tyler",32.0,0.921493,95.205000,NaN,10.488333,11.771667,18.835993,<NA>,0.125000,0.312500
1015,828353,"Garkow, Nate",30.0,0.997045,88.763636,14.563636,11.757386,11.613636,15.257566,<NA>,0.200000,0.400000
1016,828496,"Todd, Jonathan",16.0,0.947057,92.854545,NaN,8.094545,NaN,13.438236,<NA>,0.000000,0.250000


In [25]:
selected_pitchers = ["deGrom, Jacob", "Eovaldi, Nathan", "Leiter, Jack", "Corbin, Patrick", "Mahle, Tyler"]

# Filter the summary DataFrame to only include these pitchers
summary_filtered = summary[summary["player_name"].isin(selected_pitchers)]

# Round the numerical columns to two decimal places
summary_filtered_rounded = summary_filtered.round(2)

summary_filtered_rounded


,pitcher,player_name,pitches,entropy,fb_vel,gap_FB_CH,gap_FB_SL,gap_FB_CB,move_sep,xwOBA,whiff_rate,csw_rate
40,543135,"Eovaldi, Nathan",1980.0,1.55,92.51,NaN,6.76,16.67,18.03,0.28,0.13,0.29
61,571578,"Corbin, Patrick",2588.0,1.43,89.75,8.23,9.39,21.04,13.24,0.34,0.12,0.28
97,594798,"deGrom, Jacob",2684.0,1.13,97.47,7.70,7.12,16.54,19.56,0.29,0.14,0.29
243,641816,"Mahle, Tyler",1437.0,1.19,90.68,NaN,7.27,NaN,15.03,0.33,0.10,0.26
755,683004,"Leiter, Jack",2668.0,1.49,97.05,6.30,9.43,15.11,17.71,0.34,0.11,0.25
